# Ruby Envelope Vault — Verified main reviewed application snapshot
**End-to-end Kaggle & Google Colab Demo · English / Tiếng Việt**

Rails 8 · Ruby 3.4 · PostgreSQL 17 · Redis · AES-256-GCM · Minitest · Selenium

### English

This notebook runs the actual Ruby Envelope Vault application from its **reviewed post-v1.0.0 application-source commit `de9f7ce` (pinned by full SHA)**. It includes an account-scoped Security Activity Center, date/outcome filters, an Audit timeline, in-place Favorites and shared toast notifications. It verifies source integrity, installs dependencies using the repository's bootstrap, runs Rails and Selenium tests, creates encrypted synthetic records, and starts a live Rails web application. Screenshot capture and showcase packaging are enabled by default.

**Browser demo:** After Rails passes its local HTTP health checks, the notebook opens a **temporary Cloudflare Quick Tunnel** and displays a clickable URL. No Cloudflare email or OTP is required, but **Rails login remains mandatory**.

**Security:** Use synthetic data only. Do not upload production secrets, KEKs, database dumps or notebook outputs containing the demo password. This is a temporary development demonstration, **not production acceptance**. Public GitHub cloning requires no token; private forks require authorized read access.

### Tiếng Việt

Notebook này chạy ứng dụng Ruby Envelope Vault thực tế từ **commit `de9f7ce` trên nhánh `main` (ghim bằng SHA đầy đủ), chứa các cải tiến UI sau tag `v1.0.0`**. Snapshot mới còn hỗ trợ Favorite tại chỗ và toast dùng chung. Quy trình xác minh mã nguồn, cài đặt môi trường bằng bootstrap sẵn có, chạy kiểm thử Rails và Selenium, tạo bản ghi giả đã mã hóa và khởi động website Rails. Chức năng chụp ảnh giao diện và đóng gói showcase được bật mặc định.

**Mở demo qua trình duyệt:** Sau khi Rails vượt qua kiểm tra HTTP nội bộ, notebook khởi chạy **Cloudflare Quick Tunnel tạm thời** và hiển thị liên kết có thể bấm. Không cần email hay OTP Cloudflare, nhưng **vẫn phải đăng nhập Rails**.

**Bảo mật:** Chỉ sử dụng dữ liệu giả. Không tải secret thật, KEK, database dump hoặc output chứa mật khẩu demo lên GitHub. Đây là môi trường trình diễn tạm thời, **không phải nghiệm thu production**. Repo Public không cần token; private fork cần quyền đọc hợp lệ.

**Repository:** https://github.com/dangkhoa2016/Rails-8-Envelope-Encryption-Vault

**Scope / Phạm vi:** This is a pinned application-source commit prepared for a release after `v1.0.0`, not the release tag object itself. / Đây là commit ứng dụng được phát triển sau tag `v1.0.0`, không phải object tag phát hành.


## 0. Configuration
*Cấu hình*

**English**

Use **Kaggle CPU with Internet enabled** or **Google Colab CPU**. A GPU is not required. Keep the runtime connected while viewing the web app. Each Run All uses an independent Rails HTTP port and PostgreSQL demo database to avoid an old app masquerading as the new release. Each complete Run All uses a fresh, isolated source checkout; previously generated checkouts are preserved until the runtime is reset, so repeated runs consume extra disk space.

The settings cell below uses these defaults:

- `ENABLE_CLOUDFLARE_TUNNEL = True` — create a temporary browser URL **after Rails becomes healthy**; set `False` for localhost only.
- `CAPTURE_SCREENSHOTS = True` — capture ten real desktop/mobile screenshots.
- `MAKE_SHOWCASE_PACKAGE = True` — build and verify the sanitized showcase package.
- `REF = "main"` + `EXPECTED_SOURCE_COMMIT` — clone full branch history, then check out the exact reviewed commit.

The temporary URL is publicly reachable, although Rails authentication is still required. Use synthetic data and keep the demo password private.

**Tiếng Việt**

Chọn **Kaggle CPU (bật Internet)** hoặc **Google Colab CPU**. Không cần GPU. Giữ runtime hoạt động trong lúc mở giao diện web. Mỗi lượt Run All dùng cổng HTTP Rails và database PostgreSQL demo riêng, tránh nhận nhầm server cũ là bản mới. Mỗi lần Run All tạo checkout mã nguồn sạch và độc lập; các checkout cũ được giữ nguyên cho đến khi reset runtime, nên chạy nhiều lần sẽ tăng dung lượng lưu trữ.

Cell cấu hình bên dưới có các giá trị mặc định:

- `ENABLE_CLOUDFLARE_TUNNEL = True` — tạo URL trình duyệt tạm thời **sau khi Rails hoạt động ổn định**; đặt `False` nếu chỉ chạy localhost.
- `CAPTURE_SCREENSHOTS = True` — chụp mười ảnh giao diện desktop/mobile thực tế.
- `MAKE_SHOWCASE_PACKAGE = True` — đóng gói và xác minh showcase đã loại dữ liệu nhạy cảm.
- `REF = "main"` + `EXPECTED_SOURCE_COMMIT` — tải lịch sử nhánh main, rồi checkout đúng commit đã kiểm tra.

URL tạm thời có thể truy cập công khai, dù Rails vẫn yêu cầu đăng nhập. Chỉ dùng dữ liệu giả và giữ kín mật khẩu demo.


In [ ]:
from pathlib import Path
import os, sys, re, time, json, secrets, socket, subprocess, tempfile, urllib.request

REPO_SLUG = "dangkhoa2016/Rails-8-Envelope-Encryption-Vault"
REF = "main"
EXPECTED_SOURCE_COMMIT = "de9f7cef10fbc564dca57c19b021b8b8e4f9501c"
EXPECTED_SOURCE_TREE = "588a8eb98fecde5750a6c431e812b5e3bd1f3d43"
SOURCE_KIND = "PINNED_MAIN_COMMIT"
BASE = Path("/kaggle/working" if Path("/kaggle/working").is_dir() else "/content")
RUN_ID = secrets.token_hex(6)
REPO = BASE / f"ruby-envelope-vault-main-demo-{RUN_ID}"
DB_USER = f"vault_demo_{RUN_ID}"
DB_NAME = f"vault_demo_db_{RUN_ID}"
TEST_DB_NAME = f"vault_demo_test_{RUN_ID}"
RUNTIME = BASE / ".ruby-envelope-vault-runtime"
ARTIFACTS = BASE / "ruby-envelope-vault-demo-evidence"
ARTIFACTS.mkdir(parents=True, exist_ok=True, mode=0o700)
ARTIFACTS.chmod(0o700)
# Never use an HTTP port still occupied by a previous Rails server.
with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as port_probe:
    port_probe.bind(("127.0.0.1", 0))
    PORT = port_probe.getsockname()[1]
ENABLE_CLOUDFLARE_TUNNEL = True  # temporary public Quick Tunnel; Rails login remains required
CAPTURE_SCREENSHOTS = True
MAKE_SHOWCASE_PACKAGE = True
DEMO_EMAIL = "showcase@example.test"
print("Platform:", "Kaggle" if str(BASE)=="/kaggle/working" else "Colab")
print("Source:", REPO_SLUG, REF)
print("Fresh checkout:", REPO)
print("Isolated demo database:", DB_NAME, "HTTP port:", PORT)
print("Evidence directory:", ARTIFACTS)

## 1. Clone and verify the GitHub source
*Tải và xác minh mã nguồn GitHub*

**English**

The notebook clones the repository's **`main` history**, checks out the **exact reviewed commit `de9f7cef10fbc564dca57c19b021b8b8e4f9501c`** in detached HEAD, and rejects a mismatched source tree. Public repositories can be cloned anonymously; private forks require authorized access and a read-only `GH_READ_TOKEN` with **Contents: Read** permission.

The token is retrieved from Kaggle Secrets, Colab Secrets or the environment; if absent, it is requested privately. A temporary `GIT_ASKPASS` helper keeps it out of clone URLs and the Git history.

Before executing the application, the notebook confirms the pinned commit is reachable from `origin/main`, verifies both commit and tree SHA, checks Git object integrity with `git fsck`, checks application VERSION and requires a clean working tree. A later change to `main` does not silently change the executed source. A fresh per-run directory prevents a previously used checkout from causing a false dirty-working-tree failure; existing source directories are never automatically deleted or overwritten. A commit SHA and clean Git objects establish a reproducible source snapshot, not independent publisher authenticity. Use trusted provenance or a signed reference to establish authenticity.

**Tiếng Việt**

Notebook tải lịch sử **nhánh `main`**, sau đó checkout **đúng commit `de9f7cef10fbc564dca57c19b021b8b8e4f9501c`** ở chế độ detached HEAD. Repository public có thể tải ẩn danh; private fork cần quyền truy cập hợp lệ và `GH_READ_TOKEN` chỉ có quyền **Contents: Read**.

Token được lấy từ Kaggle Secrets, Colab Secrets hoặc biến môi trường; nếu thiếu, notebook yêu cầu nhập riêng tư. Công cụ `GIT_ASKPASS` tạm thời tránh ghi token vào URL clone và lịch sử Git.

Trước khi chạy ứng dụng, notebook xác minh commit nằm trong lịch sử `origin/main`, đối chiếu SHA commit và tree, chạy `git fsck`, xác minh VERSION và working tree sạch. Mỗi lượt chạy sử dụng thư mục checkout mới để tránh lỗi working tree bẩn do lần chạy trước; không tự động xóa hoặc ghi đè mã nguồn cũ. SHA cố định bảo đảm tính tái lập nhưng không tự chứng minh danh tính tác giả; cần nguồn provenance tin cậy hoặc chữ ký để xác thực độc lập.


In [ ]:
def read_github_token():
    try:
        from kaggle_secrets import UserSecretsClient
        value = UserSecretsClient().get_secret("GH_READ_TOKEN")
        if value:
            return value
    except Exception:
        pass
    try:
        from google.colab import userdata
        value = userdata.get("GH_READ_TOKEN")
        if value:
            return value
    except Exception:
        pass
    value = os.environ.get("GH_READ_TOKEN", "")
    if value:
        return value
    from getpass import getpass
    return getpass("GitHub private repo READ-ONLY token (input hidden): ")

def classify_git_clone_failure(stderr):
    """Categorize Git stderr without printing Git output or credentials."""
    message = (stderr or "").lower()
    if any(text in message for text in ("could not resolve host", "connection timed out",
                                       "failed to connect", "network is unreachable",
                                       "gnutls_handshake", "ssl certificate problem")):
        return "NETWORK_OR_TLS"
    if any(text in message for text in ("remote branch", "couldn't find remote ref",
                                       "not found in upstream origin")):
        return "MAIN_BRANCH_NOT_FOUND"
    if any(text in message for text in ("authentication failed", "invalid username or password",
                                       "http basic: access denied", "http 401",
                                       "http 403", "the requested url returned error: 403",
                                       "could not read username", "terminal prompts disabled")):
        return "AUTHENTICATION_OR_PERMISSION"
    if any(text in message for text in ("repository not found", "error: 404",
                                       "the requested url returned error: 404")):
        return "PRIVATE_REPOSITORY_NOT_ACCESSIBLE"
    return "UNKNOWN_GIT_ERROR"

def git(*args, cwd=None, env=None):
    return subprocess.run(["git", *args], cwd=cwd, env=env, text=True,
                          stdout=subprocess.PIPE, stderr=subprocess.PIPE, check=True).stdout.strip()

if not (REPO / ".git").is_dir():
    if REPO.exists():
        raise RuntimeError(f"Refusing to overwrite an existing non-Git directory: {REPO}")
    clone = ["git", "-c", "credential.helper=", "clone",
             "--branch", REF, "--single-branch",
             f"https://github.com/{REPO_SLUG}.git", str(REPO)]
    anonymous = subprocess.run(clone, stdout=subprocess.DEVNULL,
                               stderr=subprocess.PIPE, text=True,
                               env={**os.environ, "GIT_TERMINAL_PROMPT": "0"})
    if anonymous.returncode:
        # Public repositories work anonymously; private ones need read access.
        if REPO.exists():
            raise RuntimeError("Clone left an incomplete directory; remove it before retrying")
        anonymous_reason = classify_git_clone_failure(anonymous.stderr)
        if anonymous_reason == "NETWORK_OR_TLS":
            raise RuntimeError(
                "GITHUB_CLONE=FAIL reason=NETWORK_OR_TLS: Check Colab Internet/DNS/TLS connectivity."
            )
        token = read_github_token().strip()
        if not token:
            raise RuntimeError(
                "GITHUB_CLONE=FAIL reason=NO_TOKEN: Private repository requires GH_READ_TOKEN "
                "and authorized GitHub Contents: Read access."
            )
        with tempfile.TemporaryDirectory(prefix="rv-git-auth-", dir=str(BASE)) as authdir:
            askpass = Path(authdir) / "askpass.sh"
            askpass.write_text('#!/bin/sh\ncase "$1" in *Username*) printf "%s" "x-access-token";; *Password*) printf "%s" "$RV_GITHUB_TOKEN";; *) exit 1;; esac\n')
            askpass.chmod(0o700)
            env = os.environ.copy()
            env.update({"RV_GITHUB_TOKEN": token, "GIT_ASKPASS": str(askpass),
                        "GIT_TERMINAL_PROMPT": "0", "GIT_CONFIG_NOSYSTEM": "1"})
            try:
                result = subprocess.run(
                    clone, env=env, stdout=subprocess.DEVNULL,
                    stderr=subprocess.PIPE, text=True, check=False
                )
                if result.returncode != 0:
                    reason = classify_git_clone_failure(result.stderr)
                    raise RuntimeError(
                        f"GITHUB_CLONE=FAIL reason={reason} git_exit={result.returncode}. "
                        "Verify that GH_READ_TOKEN is valid and permitted to read "
                        "dangkhoa2016/Rails-8-Envelope-Encryption-Vault "
                        "(fine-grained token: selected repository + Contents: Read; "
                        "authorize pending organization/SSO access if applicable). "
                        "For NETWORK_OR_TLS, check Colab Internet. "
                        "The raw Git error is intentionally not shown to protect credentials. "
                        "If an incomplete checkout was created, restart in a fresh runtime."
                    )
            finally:
                env.pop("RV_GITHUB_TOKEN", None)
                token = None
# The full branch history includes the pinned commit even after newer documentation
# changes land on main. A shallow `--depth 1` clone would lose this guarantee.
subprocess.run(["git", "checkout", "--detach", EXPECTED_SOURCE_COMMIT], cwd=REPO,
               stdout=subprocess.DEVNULL, stderr=subprocess.PIPE, text=True, check=True)
sha = git("rev-parse", "HEAD", cwd=REPO)
if sha != EXPECTED_SOURCE_COMMIT:
    raise RuntimeError("SOURCE_COMMIT_MISMATCH: refusing to execute unreviewed code")
commit_tree = git("rev-parse", "HEAD^{tree}", cwd=REPO)
if commit_tree != EXPECTED_SOURCE_TREE:
    raise RuntimeError("SOURCE_TREE_MISMATCH: refusing unexpected source tree")
subprocess.run(["git", "merge-base", "--is-ancestor", EXPECTED_SOURCE_COMMIT,
                "refs/remotes/origin/main"], cwd=REPO, check=True,
               stdout=subprocess.DEVNULL, stderr=subprocess.PIPE)
if len(git("rev-list", "--parents", "-n", "1", "HEAD", cwd=REPO).split()) != 2:
    raise RuntimeError("PINNED_SOURCE_EXPECTS_SINGLE_PARENT")
# Verify Git object checksums and object connectivity; shallow clones are supported.
git("fsck", "--full", "--strict", "--no-reflogs", "--no-progress", cwd=REPO)
if (REPO / "VERSION").read_text().strip() != "1.0.0":
    raise RuntimeError("Application VERSION mismatch")
source_status = git("status", "--porcelain", cwd=REPO)
if source_status:
    raise RuntimeError(
        "SOURCE_CHECKOUT=FAIL reason=DIRTY_WORKTREE: Fresh pinned checkout unexpectedly "
        f"contains {len(source_status.splitlines())} changed/untracked paths. "
        "Do not delete or reset existing files automatically. Start a fresh Run All "
        "from the settings cell to create a new isolated checkout."
    )
print("SOURCE_VERIFIED=PASS", sha)
print("SOURCE_TREE_VERIFIED=PASS", commit_tree)
print("SOURCE_KIND=" + SOURCE_KIND)

## 2. Bootstrap and run complete automated tests
*Cài đặt môi trường và chạy đầy đủ kiểm thử*

**English**

The notebook executes the repository's existing `bootstrap.sh`. It prepares Ruby, PostgreSQL, Redis, dependencies, databases and a **temporary demo KEK**, then runs crypto/security self-checks and environment diagnostics.

It separately executes the **full Rails unit/integration and Selenium system-test suites** with `umask 022` for the test subprocess only. The application contains a directory-permission test sensitive to the inherited umask; the bootstrap keeps a stricter umask for protected files. This isolated test setting does not relax secret-file permissions.

Initial installation may take time. **Do not start this cell again while it is running.** If a stage fails, inspect its protected log and redact sensitive values before sharing excerpts.

**Tiếng Việt**

Notebook sử dụng trực tiếp `bootstrap.sh` của dự án để chuẩn bị Ruby, PostgreSQL, Redis, dependency, database và **KEK tạm thời cho demo**, sau đó chạy các kiểm tra crypto/security và chẩn đoán môi trường.

Tiếp theo, notebook chạy **toàn bộ Rails unit/integration và Selenium system tests** với `umask 022` chỉ dành cho tiến trình test. Một bài test của bản phát hành phụ thuộc umask được kế thừa, trong khi bootstrap chủ động dùng umask nghiêm ngặt để bảo vệ file bí mật. Thiết lập riêng cho test không nới lỏng quyền của các file này.

Lần cài đầu có thể mất thời gian. **Không chạy lại cell khi lần chạy trước chưa kết thúc.** Nếu lỗi, xem log được chỉ định và che giá trị nhạy cảm trước khi chia sẻ.


In [ ]:
BOOTSTRAP_LOG = ARTIFACTS / "bootstrap.log"
base_env = os.environ.copy()
# Credentials from earlier sessions must never overwrite another demo role.
for stale in ("DATABASE_URL", "TEST_DATABASE_URL", "POSTGRES_PASSWORD",
              "POSTGRES_APP_USER", "POSTGRES_APP_DB", "POSTGRES_TEST_DB"):
    base_env.pop(stale, None)
base_env.update({"POSTGRES_APP_USER":DB_USER, "POSTGRES_APP_DB":DB_NAME,
                 "POSTGRES_TEST_DB":TEST_DB_NAME,
                 "START_SERVER":"0", "RUN_TESTS":"0", "CREATE_OWNER":"0",
                 "ALLOW_SIGNUPS":"false", "DEMO_MODE":"true", "REQUIRE_MFA":"false",
                 "REAUTH_ON_REVEAL":"true", "PORT":str(PORT), "INSTALL_AGE":"1"})
# Reuse the managed Redis daemon's private credential without printing it.
# PostgreSQL credentials are isolated by the distinct role/database above.
redis_conf = RUNTIME / "redis" / "redis.conf"
if redis_conf.is_file() and not redis_conf.is_symlink():
    import stat
    if stat.S_IMODE(redis_conf.stat().st_mode) & 0o077:
        raise RuntimeError("Private Redis configuration has unsafe permissions")
    prior_redis_password = re.search(
        r"(?m)^requirepass ([A-Za-z0-9._-]+)$",
        redis_conf.read_text(encoding="utf-8"))
    if prior_redis_password:
        base_env["REDIS_PASSWORD"] = prior_redis_password.group(1)
    del prior_redis_password
with BOOTSTRAP_LOG.open("w", encoding="utf-8") as log:
    os.chmod(BOOTSTRAP_LOG, 0o600)
    cp = subprocess.run(["bash", "./bootstrap.sh", "setup"], cwd=REPO,
                        env=base_env, stdout=log, stderr=subprocess.STDOUT,
                        stdin=subprocess.DEVNULL)
if cp.returncode:
    raise RuntimeError(f"Bootstrap failed with exit={cp.returncode}. Examine PRIVATE log: {BOOTSTRAP_LOG}")
print("BOOTSTRAP=PASS", "crypto_security_doctor=completed", "log:", BOOTSTRAP_LOG)

# Full Rails + Selenium suites remain mandatory: controlled test-only umask 022.
# The permission test assumes a 0755 Dir.mkdir output; use 022 for tests only.
TESTS_LOG = ARTIFACTS / "full-tests.log"
test_script = """source ./activate-runtime.sh
umask 022
RAILS_ENV=test bin/rails zeitwerk:check &&
RAILS_ENV=test bin/rails db:prepare &&
RAILS_ENV=test bin/rails test &&
RAILS_ENV=test bin/rails test:system
"""
with TESTS_LOG.open("w",encoding="utf-8") as log:
    os.chmod(TESTS_LOG,0o600)
    tests = subprocess.run(["bash","-e","-c",test_script],cwd=REPO,env=base_env,
                           stdout=log,stderr=subprocess.STDOUT,stdin=subprocess.DEVNULL)
if tests.returncode:
    raise RuntimeError(f"Full Rails/Selenium test run failed (exit={tests.returncode}); private log: {TESTS_LOG}")
print("FULL_RAILS_TESTS=PASS", "RAILS_SYSTEM_TESTS=PASS", "test_umask=022", "log:", TESTS_LOG)

## 3. Create encrypted demo fixtures
*Tạo dữ liệu demo đã mã hóa*

**English**

A random temporary Rails password (at least 14 characters) is stored **outside Git** in a protected `0600` file. The application's `demo:fixture` task creates seven synthetic vault categories: **Login, API Key, Environment, Database, Secure Note, SSH and Generic**, including encrypted versions and audit events.

Sign in as `showcase@example.test` using the password printed in **your private notebook output**. New-account signups are disabled.

**Never share the demo password or publish executed notebook outputs.** Clear output cells before distributing any notebook copy.

**Tiếng Việt**

Notebook tạo mật khẩu Rails tạm thời và ngẫu nhiên (ít nhất 14 ký tự), lưu **ngoài Git** trong file có quyền `0600`. Tác vụ `demo:fixture` tạo bảy nhóm dữ liệu giả: **Login, API Key, Environment, Database, Secure Note, SSH và Generic**, cùng các phiên bản đã mã hóa và sự kiện audit.

Đăng nhập bằng `showcase@example.test` và mật khẩu trong **output notebook riêng tư của bạn**. Chức năng tự đăng ký tài khoản mới đã tắt.

**Không chia sẻ mật khẩu demo hoặc công bố notebook kèm output đã chạy.** Xóa output trước khi phát hành bản sao notebook.


In [ ]:
password_file = RUNTIME / "tmp" / "notebook-demo-password"
password_file.parent.mkdir(parents=True, exist_ok=True, mode=0o700)
password_file.parent.chmod(0o700)
if password_file.exists():
    demo_password = password_file.read_text().strip()
else:
    demo_password = secrets.token_urlsafe(22)
    fd = os.open(password_file, os.O_WRONLY | os.O_CREAT | os.O_EXCL, 0o600)
    with os.fdopen(fd, "w") as f:
        f.write(demo_password)
password_file.chmod(0o600)
if len(demo_password) < 14:
    raise RuntimeError("Demo password too short")
fixture_env = base_env.copy()
fixture_env.update({"DEMO_MODE":"true", "ALLOW_SIGNUPS":"false", "DEMO_PASSWORD_FILE":str(password_file),
                    "DEMO_EMAIL":DEMO_EMAIL, "DEMO_RESET":"false"})
fixture_cmd = 'source ./activate-runtime.sh && exec bin/rails demo:fixture'
cp = subprocess.run(["bash", "-c", fixture_cmd], cwd=REPO, env=fixture_env,
                    stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
if cp.returncode or "DEMO_FIXTURE=PASS" not in cp.stdout:
    raise RuntimeError("Fixture failed; inspect runtime configuration and Rails environment privately")
print(cp.stdout.strip())  # safe fixture summary only: no vault secret contents
print("LOGIN_EMAIL:", DEMO_EMAIL)
print("TEMPORARY_DEMO_PASSWORD (PRIVATE NOTEBOOK OUTPUT; CLEAR BEFORE SHARING):", demo_password)

## 4. Prepare browser access
*Chuẩn bị truy cập từ trình duyệt*

**English**

This cell only initializes the browser-access settings; **it does not start Cloudflare yet**. The next stage starts Rails first and waits for both `/up` and `/session/new` to return HTTP 200.

Once Rails is ready, the notebook can run:

`cloudflared tunnel --no-autoupdate --url http://127.0.0.1:3000`

There is **no Cloudflare email or OTP prompt**. The temporary public URL opens the password-protected Rails login page. Use synthetic records only.

**Tiếng Việt**

Cell này chỉ chuẩn bị cấu hình truy cập; **chưa khởi chạy Cloudflare**. Giai đoạn tiếp theo phải khởi động Rails trước và chờ cả `/up` lẫn `/session/new` trả về HTTP 200.

Sau khi Rails sẵn sàng, notebook mới chạy lệnh `cloudflared` ở trên để tạo URL tạm thời.

**Không yêu cầu email hoặc OTP Cloudflare**. URL công khai tạm thời dẫn đến trang đăng nhập Rails có mật khẩu. Chỉ sử dụng dữ liệu giả.


In [ ]:
# Deliberately start Cloudflare only AFTER Rails local HTTP is ready (cell 12).
# Do not reuse a prior Run All's Cloudflare URL for the new server.
tunnel_proc = None
tunnel_url = None
print("CLOUDFLARE_TUNNEL:", "ON (after Rails health)" if ENABLE_CLOUDFLARE_TUNNEL else "OFF (local-only)")


## 5. Start Rails, then open the temporary URL
*Khởi động Rails trước khi mở URL tạm thời*

**English**

This cell follows a fixed sequence:

1. Start Rails through `bootstrap.sh start`.
2. Confirm `GET /up` returns **HTTP 200**.
3. Confirm `GET /session/new` returns **HTTP 200**.
4. If enabled, start Cloudflare Quick Tunnel and retrieve its URL.
5. Display the clickable **`BROWSER_DEMO_URL`**.

Open the link in another tab. Sign in with `showcase@example.test` and the generated demo password. **No Cloudflare OTP is needed.** Keep the notebook runtime active while browsing.

**Tiếng Việt**

Cell thực hiện đúng trình tự:

1. Khởi động Rails bằng `bootstrap.sh start`.
2. Xác nhận `GET /up` trả về **HTTP 200**.
3. Xác nhận `GET /session/new` trả về **HTTP 200**.
4. Nếu đã bật, khởi chạy Cloudflare Quick Tunnel và lấy URL.
5. Hiển thị liên kết **`BROWSER_DEMO_URL`** có thể bấm.

Mở liên kết ở tab khác, đăng nhập bằng `showcase@example.test` và mật khẩu demo đã tạo. **Không cần OTP Cloudflare.** Giữ phiên notebook hoạt động khi duyệt website.


In [ ]:
rails_log = ARTIFACTS / "rails.log"
rails_proc = globals().get("rails_proc")
server_identity = (str(REPO), PORT)
if globals().get("_rails_server_identity") != server_identity:
    rails_proc = None  # Never reuse an older source tree's Rails Popen object.
    _rails_server_identity = server_identity
if rails_proc is None or rails_proc.poll() is not None:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as port_probe:
        if port_probe.connect_ex(("127.0.0.1", PORT)) == 0:
            raise RuntimeError("RAILS_START=FAIL reason=PORT_ALREADY_IN_USE")
    rails_env = fixture_env.copy()
    rails_env.update({"START_SERVER":"1", "PORT":str(PORT)})
    if ENABLE_CLOUDFLARE_TUNNEL:
        # Rails Host Authorization must accept short-lived trycloudflare.com hosts.
        rails_env["RAILS_DEVELOPMENT_HOSTS"] = ".trycloudflare.com"
    f=rails_log.open("w", encoding="utf-8")
    os.chmod(rails_log, 0o600)
    rails_proc=subprocess.Popen(["bash", "./bootstrap.sh", "start"], cwd=REPO,
                                env=rails_env, stdout=f, stderr=subprocess.STDOUT,
                                stdin=subprocess.DEVNULL, start_new_session=True)
    f.close()
local_url=f"http://127.0.0.1:{PORT}"
health=None
for _ in range(100):
    try:
        with urllib.request.urlopen(local_url+"/up",timeout=3) as x:
            health=x.status
        if health==200: break
    except Exception:
        pass
    if rails_proc.poll() is not None:
        raise RuntimeError(f"Rails exited unexpectedly. Check protected log: {rails_log}")
    time.sleep(2)
if health!=200 or rails_proc.poll() is not None:
    raise RuntimeError(f"Local Rails health failed or server exited; inspect protected log: {rails_log}")
# /up can be HTTP 200 even when database authentication is broken.
# Readiness must use the same newly bootstrapped environment.
db_check = subprocess.run(
    ["bash", "-e", "-c",
     """source ./activate-runtime.sh >/dev/null
bin/rails runner 'abort("DB_CONNECTION_FAIL") unless ActiveRecord::Base.connection.select_value("SELECT 1").to_i == 1'
"""],
    cwd=REPO, env=rails_env, stdout=subprocess.PIPE,
    stderr=subprocess.PIPE, text=True, stdin=subprocess.DEVNULL)
if db_check.returncode:
    raise RuntimeError("RAILS_DB_READINESS=FAIL reason=DATABASE_AUTH_OR_CONNECTIVITY; "
                       f"inspect protected Rails log: {rails_log}")
print("RAILS_DB_READINESS=PASS authenticated PostgreSQL SELECT 1")
with urllib.request.urlopen(local_url+"/session/new",timeout=10) as resp:
    login_status=resp.status
    login_html=resp.read(200_000).decode("utf-8","replace")
assert login_status==200 and ("password" in login_html.lower())
print("RAILS_HEALTH=PASS /up=200")
print("LOGIN_FORM=PASS /session/new=200")
print("LOCAL_URL:",local_url+"/session/new")

if ENABLE_CLOUDFLARE_TUNNEL:
    import platform
    cloudflared = RUNTIME / "bin" / "cloudflared"
    cloudflared.parent.mkdir(parents=True, exist_ok=True)
    if not cloudflared.is_file():
        if platform.machine().lower() not in ("x86_64", "amd64"):
            raise RuntimeError("cloudflared automatic download currently supports linux-amd64 only")
        dl_url = "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"
        temp_binary = cloudflared.with_suffix(".partial")
        urllib.request.urlretrieve(dl_url, temp_binary)
        if temp_binary.open("rb").read(4) != b"\x7fELF":
            raise RuntimeError("Downloaded cloudflared does not have a valid ELF header")
        temp_binary.chmod(0o700)
        temp_binary.replace(cloudflared)
    cloudflared.chmod(0o700)
    if tunnel_proc is None or tunnel_proc.poll() is not None or not tunnel_url:
        tunnel_log = ARTIFACTS / "cloudflared.log"
        fh = tunnel_log.open("w", encoding="utf-8")
        os.chmod(tunnel_log, 0o600)
        tunnel_proc = subprocess.Popen(
            [str(cloudflared), "tunnel", "--no-autoupdate", "--url", local_url],
            stdout=fh, stderr=subprocess.STDOUT,
            stdin=subprocess.DEVNULL, start_new_session=True)
        fh.close()
        tunnel_url = None
        pattern = re.compile(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com")
        for _ in range(60):
            found = pattern.search(tunnel_log.read_text(errors="replace"))
            if found:
                tunnel_url = found.group(0)
                break
            if tunnel_proc.poll() is not None:
                raise RuntimeError(f"cloudflared exited. Inspect private log: {tunnel_log}")
            time.sleep(2)
        if not tunnel_url:
            tunnel_proc.terminate()
            raise RuntimeError(f"No Cloudflare Quick Tunnel URL. Inspect private log: {tunnel_log}")
    from IPython.display import display, HTML
    browser_url = tunnel_url + "/session/new"
    print("CLOUDFLARE_TUNNEL=READY (temporary public demo; Rails login required)")
    print("BROWSER_DEMO_URL:", browser_url)
    display(HTML('<a href="' + browser_url + '" target="_blank" rel="noopener noreferrer">Open Ruby Envelope Vault demo in browser</a>'))
else:
    tunnel_url = None
    print("TUNNEL=OFF (local-only mode explicitly selected)")


## 6. Inspect encrypted database records
*Kiểm tra bản ghi mã hóa trong database*

**English**

This stage reads **non-secret counts and envelope metadata only**. It requires at least four items in **each of the seven supported categories** and at least 29 immutable versions, then checks the synthetic records for ciphertext, wrapped data-encryption keys (DEKs), **12-byte GCM nonces** and **16-byte authentication tags**. No secret plaintext is printed.

`ENCRYPTED_FIXTURES=PASS` confirms the stored envelope structure meets these checks. It is not an independent cryptographic audit or a production recovery test.

**Tiếng Việt**

Giai đoạn này chỉ đọc **số liệu không nhạy cảm và metadata envelope**. Notebook yêu cầu ít nhất bốn item thuộc **mỗi loại trong bảy loại** và ít nhất 29 phiên bản bất biến, sau đó kiểm tra ciphertext, DEK đã bọc khóa, **nonce GCM dài 12 byte** và **authentication tag dài 16 byte** của các bản ghi giả. Không in secret dạng rõ.

`ENCRYPTED_FIXTURES=PASS` xác nhận cấu trúc envelope đã lưu đáp ứng các kiểm tra trên, không thay thế kiểm toán mật mã độc lập hoặc kiểm thử phục hồi production.


In [ ]:
ruby_src = r"""
require "json"
user = User.find_by!(email_address: ENV.fetch("DEMO_EMAIL"))
items = user.vault_items.includes(:versions).to_a
versions = items.flat_map(&:versions)
# Verify every supported category, not a stale six-item fixture snapshot.
kind_counts = items.group_by(&:kind).transform_values(&:size)
raise "incomplete demo fixture coverage" unless
  VaultItem::KINDS.all? { |kind| kind_counts.fetch(kind, 0) >= 4 }
raise "missing immutable demo versions" unless versions.size >= 29
raise "invalid ciphertext envelope" unless versions.all? { |v|
  v.cipher_suite == "AES-256-GCM" && v.nonce.bytesize == 12 &&
  v.authentication_tag.bytesize == 16 && v.ciphertext.bytesize > 0 &&
  v.wrapped_dek.bytesize > 0
}
puts "ENCRYPTED_FIXTURES=PASS #{JSON.generate(items: items.size, versions: versions.size,
  kinds: items.map(&:kind).sort, cipher: 'AES-256-GCM', aead_nonce_bytes: 12, auth_tag_bytes: 16)}"
"""
with tempfile.NamedTemporaryFile(mode="w",suffix=".rb",dir=str(ARTIFACTS),delete=False) as f:
    f.write(ruby_src)
    rubyfile=Path(f.name)
rubyfile.chmod(0o600)
try:
    env = fixture_env.copy()
    proc=subprocess.run(["bash","-c",f'source ./activate-runtime.sh && exec bin/rails runner "{rubyfile}"'],
                        cwd=REPO,env=env,text=True,stdout=subprocess.PIPE,stderr=subprocess.PIPE)
    if proc.returncode: raise RuntimeError("Encrypted fixture verification failed; inspect Rails logs privately")
    print(proc.stdout.strip())
finally:
    rubyfile.unlink(missing_ok=True)

## 7. Capture real desktop and mobile screenshots
*Chụp ảnh giao diện desktop và mobile thực tế*

**English**

With `CAPTURE_SCREENSHOTS = True` (default), the repository's Selenium-based `script/capture-showcase` workflow captures **ten real browser screenshots** across desktop/mobile views. Its separate test database preserves the running development demo.

The workflow writes PNG files, checksums and a capture manifest. These are actual Rails pages, **not generated UI mockups**. Check private capture logs if browser timing causes an intermittent failure.

**Tiếng Việt**

Khi `CAPTURE_SCREENSHOTS = True` (mặc định), công cụ `script/capture-showcase` dùng Selenium để chụp **mười ảnh giao diện trình duyệt thực tế** ở chế độ desktop/mobile. Database test riêng giúp giữ nguyên dữ liệu phiên demo đang chạy.

Quy trình xuất PNG, checksum và capture manifest. Đây là trang Rails thật, **không phải ảnh mockup**. Nếu gặp lỗi thời gian tải trang, hãy kiểm tra log capture riêng tư.


In [ ]:
SCREENSHOTS = ARTIFACTS / "screenshots"
if CAPTURE_SCREENSHOTS:
    capture_log=ARTIFACTS / "showcase-capture.log"
    screen_env=fixture_env.copy()
    screen_env["SHOWCASE_OUTPUT"]=str(SCREENSHOTS)
    cmd='source ./activate-runtime.sh && exec bash ./script/capture-showcase'
    with capture_log.open("w",encoding="utf-8") as f:
        os.chmod(capture_log,0o600)
        cp=subprocess.run(["bash","-c",cmd],cwd=REPO,env=screen_env,
                          stdout=f,stderr=subprocess.STDOUT)
    if cp.returncode: raise RuntimeError(f"Screenshot capture failed; private log: {capture_log}")
    shots=sorted(SCREENSHOTS.glob("*.png"))
    if len(shots)!=10: raise RuntimeError(f"Expected 10 screenshots, received {len(shots)}")
    if not (SCREENSHOTS/"SHA256SUMS.txt").is_file() or not (SCREENSHOTS/"MANIFEST.tsv").is_file():
        raise RuntimeError("Screenshot integrity metadata missing")
    print(f"SHOWCASE_CAPTURE=PASS screenshots={len(shots)}")
else:
    print("SHOWCASE_CAPTURE=SKIPPED (user setting)")

## 8. Package and verify the showcase
*Đóng gói và xác minh showcase*

**English**

When screenshot capture and packaging are enabled, this cell calls the repository's existing packaging and **SHA-256 integrity checks**. It packages the generated public-facing screenshots and documents and verifies the result.

`PUBLIC_SHOWCASE_PACKAGE=PASS` reports that the packaging checks completed. Do **not** include protected logs, demo passwords, KEKs, environment files or database dumps in any public bundle.

**Tiếng Việt**

Khi bật chụp ảnh và đóng gói, cell sử dụng công cụ có sẵn để tạo showcase và **kiểm tra SHA-256**. Gói gồm ảnh giao diện cùng tài liệu có thể công bố từ phiên chạy, sau đó được xác minh tính toàn vẹn.

`PUBLIC_SHOWCASE_PACKAGE=PASS` cho biết quá trình kiểm tra gói đã hoàn thành. **Không** đưa log bí mật, mật khẩu demo, KEK, file môi trường hoặc database dump vào gói công khai.


In [ ]:
SHOWCASE_OUT = ARTIFACTS / "showcase-package"
if MAKE_SHOWCASE_PACKAGE and CAPTURE_SCREENSHOTS:
    pkg_env=fixture_env.copy()
    pkg_env.update({"SHOWCASE_DIR":str(SCREENSHOTS),"PUBLIC_SHOWCASE_OUTPUT_DIR":str(SHOWCASE_OUT)})
    pkg_log=ARTIFACTS / "showcase-package.log"
    with pkg_log.open("w",encoding="utf-8") as f:
        os.chmod(pkg_log,0o600)
        cp=subprocess.run(["bash","./script/build-public-showcase"],cwd=REPO,
                          env=pkg_env,stdout=f,stderr=subprocess.STDOUT)
    if cp.returncode: raise RuntimeError(f"Package build failed; private log: {pkg_log}")
    print("PUBLIC_SHOWCASE_PACKAGE=PASS")
    print("Sanitized package outputs:", [p.name for p in SHOWCASE_OUT.iterdir() if p.is_file()])
else:
    print("PUBLIC_SHOWCASE_PACKAGE=SKIPPED")

## 9. Review the gallery and evidence
*Xem gallery và bằng chứng kiểm thử*

**English**

When screenshots are enabled, this cell displays a gallery of captured **synthetic-data UI images**. It also prints the verified source SHA, demo account, evidence directory and temporary `BROWSER_DEMO_URL` when available.

The Rails website remains active for manual testing. **Do not share an executed notebook that still contains its demo password.**

**Tiếng Việt**

Nếu bật chụp ảnh, cell hiển thị gallery từ các **ảnh giao diện chứa dữ liệu giả** đã thu thập. Output cũng cung cấp source SHA đã xác minh, tài khoản demo, thư mục bằng chứng và `BROWSER_DEMO_URL` tạm thời (nếu có).

Website Rails vẫn chạy để bạn kiểm thử thủ công. **Không chia sẻ notebook đã chạy khi output còn chứa mật khẩu demo.**


In [ ]:
if CAPTURE_SCREENSHOTS:
    from IPython.display import display
    from PIL import Image,ImageOps,ImageDraw
    thumbs=[]
    for path in sorted(SCREENSHOTS.glob("*.png")):
        im=Image.open(path).convert("RGB")
        im.thumbnail((420,300))
        tile=Image.new("RGB",(450,350),"white")
        tile.paste(im,((450-im.width)//2,8))
        ImageDraw.Draw(tile).text((12,322),path.stem[:52],fill="black")
        thumbs.append(tile)
    if thumbs:
        grid=Image.new("RGB",(450*2,350*5),"#dddddd")
        for idx,t in enumerate(thumbs): grid.paste(t,((idx%2)*450,(idx//2)*350))
        display(grid)
print("SOURCE_SHA:",sha)
print("DEMO_EMAIL:",DEMO_EMAIL)
print("LOCAL_URL:",local_url)
if tunnel_url: print("BROWSER_DEMO_URL:",tunnel_url+"/session/new")
print("PRIVATE_LOG_DIRECTORY:",ARTIFACTS)
print("Do not publish notebook cell outputs that contain demo passwords or private run logs.")

## 10. Shut down the temporary demo
*Dừng phiên demo tạm thời*

**English**

The following shutdown cell is **commented out by default**. Consequently, **Run All does not immediately close** the Rails website or Cloudflare URL you just opened.

When finished, **uncomment the shutdown commands and execute this final cell manually**. They terminate notebook-owned Rails/Cloudflare processes and ask the project's bootstrap to stop managed services. The temporary URL also stops working when the notebook runtime ends.

**Tiếng Việt**

Cell dừng dịch vụ bên dưới được **comment mặc định**. Vì vậy **Run All không tự đóng ngay** website Rails hoặc URL Cloudflare vừa tạo.

Sau khi hoàn tất kiểm thử, hãy **bỏ comment các lệnh dừng và chạy riêng cell cuối**. Các lệnh kết thúc tiến trình Rails/Cloudflare của notebook và yêu cầu bootstrap dừng dịch vụ liên quan. URL tạm thời cũng hết hiệu lực khi runtime kết thúc.


In [ ]:
# Run manually when you are finished (uncomment):
# if tunnel_proc is not None and tunnel_proc.poll() is None: tunnel_proc.terminate()
# if rails_proc is not None and rails_proc.poll() is None: rails_proc.terminate()
# subprocess.run(["bash", "./bootstrap.sh", "stop"], cwd=REPO, check=False)
# print("DEMO_STOPPED")